# KLTN — V2.3: Test-Time Augmentation (TTA) Ensemble

**Mục tiêu:** Đẩy F1 từ V2.2 = 0,7977 lên ≥ 0,81 bằng TTA. **Không cần train mới.**

## Phương pháp

Với mỗi sample test, tạo K = 5 phiên bản (augmented):
1. **Original** — không augment
2. **Light noise** σ=0,005
3. **Standard noise** σ=0,010
4. **Time-shift +3 frame** (mô phỏng cùng hành vi nhưng lệch pha)
5. **Time-shift −3 frame**

Mỗi V1.4 và V2.0 predict trên cả 5 phiên bản → average → 1 probability/model.
Cuối cùng áp dụng α tối ưu (đã grid search ở V2.2) → final prediction.

## Kỳ vọng
- F1 boost: +1-2% (lên 0,81 - 0,83)
- Std giảm thêm (do averaging giảm variance)
- Thời gian: ~15-20 phút inference, không train

## Yêu cầu trước
- Có folder `training_outputs/v1.4_stratified/` và `training_outputs/v2.0_stgcn/` với 5 checkpoint
- Có folder `training_outputs/v2.2_ensemble/log_seed*.json` để lấy α tối ưu

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
ROOT_DIR = "/content/drive/MyDrive/KLTN/Human-Reco"
V14_DIR = f"{ROOT_DIR}/training_outputs/v1.4_stratified"
V20_DIR = f"{ROOT_DIR}/training_outputs/v2.0_stgcn"
V22_DIR = f"{ROOT_DIR}/training_outputs/v2.2_ensemble"
OUT_DIR = f"{ROOT_DIR}/training_outputs/v2.3_tta"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"V1.4: {V14_DIR}")
print(f"V2.0: {V20_DIR}")
print(f"V2.2 logs: {V22_DIR}")
print(f"OUT:  {OUT_DIR}")

In [ ]:
import os, json, glob, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import load_model
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix
print("TF:", tf.__version__, "Torch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 2. Re-define ST-GCN model (để load V2.0 checkpoint)

In [ ]:
COCO_17_EDGES = [(0,1),(0,2),(1,3),(2,4),(5,0),(6,0),(5,6),(5,7),(7,9),(6,8),(8,10),
                 (5,11),(6,12),(11,12),(11,13),(13,15),(12,14),(14,16)]
V_JOINTS = 17; CENTER_JOINT = 0

def bfs_dist(adj, root):
    V = adj.shape[0]; dist = -np.ones(V, dtype=np.int32); dist[root] = 0
    fr = [root]
    while fr:
        nf = []
        for u in fr:
            for v in range(V):
                if adj[u,v] > 0 and dist[v] == -1:
                    dist[v] = dist[u] + 1; nf.append(v)
        fr = nf
    if (dist == -1).any(): dist[dist == -1] = dist.max() + 1
    return dist

def _norm(A):
    D = A.sum(0); Di = np.zeros_like(D, dtype=np.float32); Di[D > 0] = D[D > 0] ** -0.5
    return np.diag(Di) @ A @ np.diag(Di)

def build_adjacency():
    A = np.zeros((V_JOINTS, V_JOINTS), dtype=np.float32)
    for i,j in COCO_17_EDGES: A[i,j] = 1; A[j,i] = 1
    I = np.eye(V_JOINTS, dtype=np.float32)
    dist = bfs_dist(A, CENTER_JOINT)
    cp = np.zeros_like(A); cf = np.zeros_like(A)
    for i,j in COCO_17_EDGES:
        if dist[i] == dist[j]:    cp[i,j] = 1; cp[j,i] = 1
        elif dist[i] > dist[j]:   cp[i,j] = 1; cf[j,i] = 1
        else:                     cf[i,j] = 1; cp[j,i] = 1
    return np.stack([_norm(I), _norm(cp), _norm(cf)], axis=0)


class CTRGraphConv(nn.Module):
    def __init__(self, in_ch, out_ch, A, use_ctr=True):
        super().__init__()
        K, V, _ = A.shape; self.K, self.V = K, V
        self.register_buffer("A", torch.from_numpy(A).float())
        self.conv = nn.Conv2d(in_ch, out_ch * K, 1); self.use_ctr = use_ctr
        if use_ctr:
            mid = max(out_ch // 8, 8)
            self.theta = nn.Conv2d(in_ch, mid, 1); self.phi = nn.Conv2d(in_ch, mid, 1)
            self.alpha = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        N, _, T, V = x.shape
        y = self.conv(x); out_ch = y.shape[1] // self.K
        y = y.view(N, self.K, out_ch, T, V)
        out = torch.einsum("nkctv,kvw->nkctw", y, self.A)
        if self.use_ctr:
            theta = self.theta(x).mean(2); phi = self.phi(x).mean(2)
            offset = torch.tanh(torch.einsum("ncv,ncw->nvw", theta, phi))
            out = out + self.alpha * torch.einsum("nkctv,nvw->nkctw", y, offset)
        return out.sum(1)

class TemporalConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=9, stride=1):
        super().__init__()
        pad = (kernel-1)//2
        self.conv = nn.Conv2d(in_ch, out_ch, (kernel,1), padding=(pad,0), stride=(stride,1))
        self.bn = nn.BatchNorm2d(out_ch)
    def forward(self, x): return self.bn(self.conv(x))

class STGCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, A, t_stride=1, dropout=0.2):
        super().__init__()
        self.spatial = CTRGraphConv(in_ch, out_ch, A); self.bn_s = nn.BatchNorm2d(out_ch)
        self.temporal = TemporalConv(out_ch, out_ch, 9, t_stride)
        self.relu = nn.ReLU(inplace=True); self.dropout = nn.Dropout2d(dropout)
        if in_ch == out_ch and t_stride == 1: self.residual = nn.Identity()
        else: self.residual = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=(t_stride,1)),
                                             nn.BatchNorm2d(out_ch))
    def forward(self, x):
        res = self.residual(x)
        y = self.relu(self.bn_s(self.spatial(x)))
        y = self.temporal(y); y = self.dropout(y)
        return self.relu(y + res)

class STGCN(nn.Module):
    def __init__(self, in_ch=3, n_classes=2, channels=(64,128,256), dropout=0.5):
        super().__init__()
        A = build_adjacency(); c1, c2, c3 = channels
        self.data_bn = nn.BatchNorm1d(in_ch * V_JOINTS)
        self.b1 = STGCNBlock(in_ch, c1, A, 1, 0.2)
        self.b2 = STGCNBlock(c1, c2, A, 2, 0.2)
        self.b3 = STGCNBlock(c2, c3, A, 2, 0.2)
        self.dropout = nn.Dropout(dropout); self.fc = nn.Linear(c3, n_classes)
    def forward(self, x):
        N, C, T, V = x.shape
        x_bn = x.permute(0,1,3,2).contiguous().view(N, C*V, T)
        x_bn = self.data_bn(x_bn)
        x = x_bn.view(N, C, V, T).permute(0,1,3,2).contiguous()
        x = self.b1(x); x = self.b2(x); x = self.b3(x)
        x = x.mean(dim=(2,3))
        return self.fc(self.dropout(x))

def focal_loss_keras(alpha=0.25, gamma=2.0):
    def f(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        oh = tf.cast(tf.one_hot(y_true, depth=tf.shape(y_pred)[-1]), y_pred.dtype)
        p = tf.clip_by_value(y_pred, 1e-8, 1 - 1e-8)
        ce = -oh * tf.math.log(p)
        return tf.reduce_mean(tf.reduce_sum(alpha * tf.math.pow(1 - p, gamma) * ce, axis=-1))
    return f
print("Models defined")

## 3. Load data + split

In [ ]:
def regen_56(processed_root, seq=90, step=30):
    X, y, c = [], [], []
    for lbl in ['normal','shoplifting']:
        v = 0 if lbl=='normal' else 1
        for fp in sorted(glob.glob(os.path.join(processed_root, lbl, "*.csv"))):
            cid = f"{lbl}/{os.path.basename(fp)}"
            d = pd.read_csv(fp).values
            if d.shape[1] != 56: continue
            for i in range(0, len(d)-seq+1, step):
                X.append(d[i:i+seq]); y.append(v); c.append(cid)
    return np.array(X,'float32'), np.array(y,'int'), np.array(c)

X_56, y, clip_ids = regen_56(f"{ROOT_DIR}/processed_data1")
X_34 = X_56[:, :, :34].reshape(X_56.shape[0], X_56.shape[1], 17, 2)
conf = np.ones((X_34.shape[0], X_34.shape[1], 17, 1), dtype='float32')
X_skel = np.concatenate([X_34, conf], axis=-1).transpose(0, 3, 1, 2)
print(f"V1.4 input: {X_56.shape}, V2.0 input: {X_skel.shape}")

## 4. TTA augmentation functions

5 phiên bản cho mỗi sample:
- **0:** original
- **1:** noise σ=0,005
- **2:** noise σ=0,010
- **3:** time shift +3 frame (xén đầu, pad đuôi bằng frame cuối)
- **4:** time shift −3 frame (pad đầu bằng frame đầu, xén đuôi)

In [ ]:
def tta_versions_56(x, K=5):
    """Sinh K phiên bản từ x (T, 56)."""
    out = [x]                                              # 0: original
    out.append(x + np.random.normal(0, 0.005, x.shape).astype(np.float32))  # 1
    out.append(x + np.random.normal(0, 0.010, x.shape).astype(np.float32))  # 2
    # time shift +3 (drop first 3, repeat last)
    sh = np.concatenate([x[3:], np.tile(x[-1:], (3,1))], axis=0); out.append(sh)
    # time shift -3 (pad first, drop last)
    sh = np.concatenate([np.tile(x[:1], (3,1)), x[:-3]], axis=0); out.append(sh)
    return np.stack(out, 0)  # (K, T, 56)

def tta_versions_skel(x, K=5):
    """Sinh K phiên bản từ x (3, T, V)."""
    out = [x]
    noise1 = x.copy(); noise1[:2] += np.random.normal(0, 0.005, x[:2].shape).astype(np.float32)
    out.append(noise1)
    noise2 = x.copy(); noise2[:2] += np.random.normal(0, 0.010, x[:2].shape).astype(np.float32)
    out.append(noise2)
    # time shift +3 along T axis (axis 1)
    sh = np.concatenate([x[:, 3:], np.tile(x[:, -1:], (1, 3, 1))], axis=1); out.append(sh)
    sh = np.concatenate([np.tile(x[:, :1], (1, 3, 1)), x[:, :-3]], axis=1); out.append(sh)
    return np.stack(out, 0)

## 5. Predict với TTA cho từng model

In [ ]:
def predict_v14_tta(model_path, X_56, indices, K=5, batch=32, seed=42):
    np.random.seed(seed)
    model = load_model(model_path, custom_objects={'f': focal_loss_keras(), 'loss_fn': focal_loss_keras()},
                       compile=False)
    probs_avg = np.zeros((len(indices), 2), dtype=np.float32)
    for k in range(K):
        np.random.seed(seed + k)  # khác seed mỗi phiên bản để noise khác nhau
        # build K-th augmented batch
        X_aug = np.stack([tta_versions_56(X_56[idx], K=K)[k] for idx in indices])
        p = model.predict(X_aug, batch_size=batch, verbose=0)
        probs_avg += p
    return probs_avg / K

def predict_v20_tta(model_path, X_skel, indices, K=5, batch=32, seed=42):
    np.random.seed(seed)
    model = STGCN(in_ch=3, n_classes=2).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    probs_avg = np.zeros((len(indices), 2), dtype=np.float32)
    for k in range(K):
        np.random.seed(seed + k)
        X_aug = np.stack([tta_versions_skel(X_skel[idx], K=K)[k] for idx in indices]).astype(np.float32)
        probs_k = []
        with torch.no_grad():
            for i in range(0, len(X_aug), batch):
                xb = torch.from_numpy(X_aug[i:i+batch]).to(device)
                p = F.softmax(model(xb), dim=-1).cpu().numpy()
                probs_k.append(p)
        probs_avg += np.concatenate(probs_k, 0)
    return probs_avg / K

## 6. Hàm ensemble cho 1 seed với TTA

In [ ]:
def split_3way(X, y, groups, seed):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    folds = list(sgkf.split(X, y, groups))
    idx_test = folds[seed % 5][1]
    idx_val  = folds[(seed + 1) % 5][1]
    mask = np.ones(len(X), dtype=bool); mask[idx_test] = False; mask[idx_val] = False
    return np.where(mask)[0], idx_val, idx_test


def ensemble_tta_one_seed(X_56, X_skel, y, clip_ids, seed, K=5):
    print(f"\n=== SEED {seed} (TTA K={K}) ===")
    idx_tr, idx_v, idx_t = split_3way(X_56, y, clip_ids, seed)

    v14_ckpt = f"{V14_DIR}/best_seed{seed}.keras"
    v20_ckpt = f"{V20_DIR}/best_seed{seed}.pt"

    # Load α tối ưu từ V2.2
    try:
        with open(f"{V22_DIR}/log_seed{seed}.json") as f:
            alpha_v22 = json.load(f)["best_alpha"]
    except FileNotFoundError:
        alpha_v22 = 0.5
        print(f"  (Không thấy V2.2 log, dùng α=0.5)")

    # Predict V1.4 với TTA trên val + test
    p_v14_v = predict_v14_tta(v14_ckpt, X_56, idx_v, K=K, seed=seed)
    p_v14_t = predict_v14_tta(v14_ckpt, X_56, idx_t, K=K, seed=seed)
    p_v20_v = predict_v20_tta(v20_ckpt, X_skel, idx_v, K=K, seed=seed)
    p_v20_t = predict_v20_tta(v20_ckpt, X_skel, idx_t, K=K, seed=seed)

    y_v = y[idx_v]; y_t = y[idx_t]

    # Method A: dùng α từ V2.2 (giữ nguyên)
    p_test_a = alpha_v22 * p_v14_t + (1 - alpha_v22) * p_v20_t
    f1_a = f1_score(y_t, p_test_a.argmax(1), average='macro')

    # Method B: re-grid search trên val với TTA probs
    best_f1_v = 0; best_alpha_b = alpha_v22
    for ai in range(0, 21):
        a = ai / 20
        p_v = a * p_v14_v + (1 - a) * p_v20_v
        f1_v = f1_score(y_v, p_v.argmax(1), average='macro')
        if f1_v > best_f1_v:
            best_f1_v = f1_v; best_alpha_b = a
    p_test_b = best_alpha_b * p_v14_t + (1 - best_alpha_b) * p_v20_t
    f1_b = f1_score(y_t, p_test_b.argmax(1), average='macro')

    # Method C: simple average không grid search
    p_test_c = 0.5 * p_v14_t + 0.5 * p_v20_t
    f1_c = f1_score(y_t, p_test_c.argmax(1), average='macro')

    # Lấy method tốt nhất
    if f1_b >= f1_a and f1_b >= f1_c:
        best_method, best_f1, best_alpha, best_p = "B_regrid_TTA", f1_b, best_alpha_b, p_test_b
    elif f1_a >= f1_c:
        best_method, best_f1, best_alpha, best_p = "A_keep_alpha_V22", f1_a, alpha_v22, p_test_a
    else:
        best_method, best_f1, best_alpha, best_p = "C_avg_50_50", f1_c, 0.5, p_test_c

    pred = best_p.argmax(1)
    acc = accuracy_score(y_t, pred)
    prec = precision_score(y_t, pred, average='macro', zero_division=0)
    rec = recall_score(y_t, pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_t, pred)

    print(f"  Method A (α V2.2={alpha_v22:.2f}): F1 = {f1_a:.4f}")
    print(f"  Method B (re-grid α={best_alpha_b:.2f}):  F1 = {f1_b:.4f}")
    print(f"  Method C (α=0.5):              F1 = {f1_c:.4f}")
    print(f"  → Chọn: {best_method}, F1 = {best_f1:.4f}")
    return {
        "seed": seed, "K": K,
        "alpha_v22": alpha_v22, "best_alpha": best_alpha, "best_method": best_method,
        "f1_method_a": float(f1_a), "f1_method_b": float(f1_b), "f1_method_c": float(f1_c),
        "f1_tta_best": float(best_f1),
        "accuracy": float(acc), "precision": float(prec), "recall": float(rec),
        "cm": cm.tolist(), "test_size": int(len(idx_t)),
    }

## 7. Chạy 5 seed (~15-20 phút)

In [ ]:
SEEDS = [42, 123, 7, 2024, 999]
results = []
t0 = time.time()
for s in SEEDS:
    r = ensemble_tta_one_seed(X_56, X_skel, y, clip_ids, s, K=5)
    results.append(r)
    with open(f"{OUT_DIR}/log_seed{s}.json", "w") as f:
        json.dump(r, f, indent=2, ensure_ascii=False)
    print(f"  Done seed {s}, tổng {(time.time()-t0)/60:.1f} phút\n")
print(f"\nTổng: {(time.time()-t0)/60:.1f} phút")

## 8. So sánh V2.2 (no TTA) vs V2.3 (TTA)

In [ ]:
def stat(vs):
    return {"mean":float(np.mean(vs)),"std":float(np.std(vs,ddof=1)),
            "min":float(np.min(vs)),"max":float(np.max(vs))}

# V2.2 baseline
v22_f1 = {42:0.8259,123:0.7992,7:0.8128,2024:0.7709,999:0.7798}
v22_mean = np.mean(list(v22_f1.values()))
v22_std  = np.std(list(v22_f1.values()), ddof=1)

# V2.3 results
f1_tta = [r["f1_tta_best"] for r in results]
acc_tta = [r["accuracy"] for r in results]
prec_tta = [r["precision"] for r in results]
rec_tta = [r["recall"] for r in results]

summary = {
    "experiment": "V2.3 TTA Ensemble (K=5 augmentations per sample)",
    "seeds": SEEDS,
    "K": 5,
    "f1_macro": stat(f1_tta),
    "accuracy": stat(acc_tta),
    "precision": stat(prec_tta),
    "recall": stat(rec_tta),
    "v22_baseline": {"mean": float(v22_mean), "std": float(v22_std)},
    "individual_runs": results,
}

print("="*75)
print("V2.3 TTA SUMMARY (vs V2.2 no-TTA)")
print("="*75)
print(f"{'Config':<25} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-"*75)
print(f"{'V2.2 baseline (no TTA)':<25} {v22_mean:>10.4f} {v22_std:>10.4f} "
      f"{min(v22_f1.values()):>10.4f} {max(v22_f1.values()):>10.4f}")
print(f"{'V2.3 TTA (K=5)':<25} {summary['f1_macro']['mean']:>10.4f} "
      f"{summary['f1_macro']['std']:>10.4f} {summary['f1_macro']['min']:>10.4f} "
      f"{summary['f1_macro']['max']:>10.4f}")
print()
delta = summary['f1_macro']['mean'] - v22_mean
print(f"Δ mean F1: {delta:+.4f} ({delta*100:+.2f}%)")
print()
print("Per-seed:")
for r in results:
    v22 = v22_f1[r['seed']]
    d = r['f1_tta_best'] - v22
    print(f"  seed {r['seed']:>4d}: V2.2={v22:.4f} → V2.3={r['f1_tta_best']:.4f} ({d:+.4f}) "
          f"[{r['best_method']}]")

m = summary['f1_macro']['mean']
if m >= 0.85: print(f"\n🎉 V2.3 ĐẠT XUẤT SẮC F1={m:.4f}!")
elif m >= 0.82: print(f"\n🎉 V2.3 cao F1={m:.4f}!")
elif m >= 0.80: print(f"\n✓ V2.3 vượt 0.80 rõ ràng F1={m:.4f}!")
elif m > v22_mean: print(f"\n✓ V2.3 cải thiện so V2.2. F1={m:.4f}.")
else: print(f"\n~ V2.3 không cải thiện. F1={m:.4f}. Cần thử MS-G3D.")

with open(f"{OUT_DIR}/V2.3_TTA_summary.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

In [ ]:
# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(len(SEEDS)); w = 0.4
v22_vals = [v22_f1[s] for s in SEEDS]
v23_vals = [r['f1_tta_best'] for r in results]

axes[0].bar(x-w/2, v22_vals, w, label='V2.2 (no TTA)', color='mediumseagreen', edgecolor='black')
axes[0].bar(x+w/2, v23_vals, w, label='V2.3 (TTA K=5)', color='goldenrod', edgecolor='black')
axes[0].axhline(y=0.80, color='red', linestyle='--', alpha=0.5, label='Min 0,80')
axes[0].axhline(y=0.88, color='green', linestyle='--', alpha=0.5, label='Excellent 0,88')
axes[0].set_xticks(x); axes[0].set_xticklabels([f"seed={s}" for s in SEEDS])
axes[0].set_ylabel('F1-macro'); axes[0].set_title('V2.2 vs V2.3 TTA')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3); axes[0].set_ylim([0.7, 0.9])

axes[1].boxplot([v22_vals, v23_vals], labels=['V2.2', 'V2.3 TTA'], showmeans=True)
axes[1].axhline(y=0.80, color='red', linestyle='--', alpha=0.5)
axes[1].axhline(y=0.88, color='green', linestyle='--', alpha=0.5)
axes[1].set_title('Phân phối F1')
axes[1].grid(axis='y', alpha=0.3); axes[1].set_ylim([0.7, 0.9])

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/v2.3_tta_compare.png", dpi=150, bbox_inches='tight')
plt.show()

## 9. Bước tiếp

Gửi tôi `V2.3_TTA_summary.json`:

| V2.3 Mean F1 | Quyết định |
|---:|---|
| ≥ 0,85 | Xuất sắc — dừng tinh chỉnh, đi V3.0 demo |
| 0,82 - 0,84 | Vượt ngưỡng tốt — đi V3.0 |
| 0,80 - 0,81 | Vượt min ngưỡng — đi V3.0 hoặc thử V2.4 MS-G3D |
| < 0,80 | Cần V2.4 MS-G3D (kiến trúc tốt hơn ST-GCN) |